In [ ]:
import heapq
START_STATE = (1, 2, 3, 4, 0, 6, 7, 5, 8)
GOAL_STATE  = (1, 2, 3, 4, 5, 6, 7, 8, 0)
def count_misplaced_tiles(state):
    count = 0
    for i in range(9):
        if state[i] != 0 and state[i] != GOAL_STATE[i]:
            count += 1
    return count
def get_ucs_successors_by_misplaced(state):
    successors = []
    blank_idx = state.index(0)
    row, col = blank_idx // 3, blank_idx % 3
    for move, diff in [('U', -3), ('D', 3), ('L', -1), ('R', 1)]:
        if move == 'U' and row == 0: continue
        if move == 'D' and row == 2: continue
        if move == 'L' and col == 0: continue
        if move == 'R' and col == 2: continue
        new_idx = blank_idx + diff
        new_state = list(state)
        new_state[blank_idx], new_state[new_idx] = new_state[new_idx], new_state[blank_idx]
        next_state_tuple = tuple(new_state)
        cost = count_misplaced_tiles(next_state_tuple)
        successors.append((next_state_tuple, cost))
    return successors
def state_to_str(state):
    if state == START_STATE: return "S"
    if state == GOAL_STATE: return "GOAL"
    return f"N_{state.index(0)}"
def print_matrix(state):
    print(f"[{state[0]} {state[1]} {state[2]}]")
    print(f"[{state[3]} {state[4]} {state[5]}]")
    print(f"[{state[6]} {state[7]} {state[8]}]\n")
def print_row_ucs(action, frontier, reached, note=""):
    f_str = "[" + ", ".join([f"{state_to_str(x[1])}({x[0]})" for x in sorted(frontier)]) + "]"
    r_str = "{" + ", ".join([f"{state_to_str(k)}:{v}" for k, v in reached.items()]) + "}"
    print(f"{action:<18} | {f_str:<35} | {r_str:<30} | {note}")
def ucs_misplaced_trace(start):
    reached = {start: 0}
    frontier = []
    heapq.heappush(frontier, (0, start, [start]))
    print("-" * 115)
    print(f"{'Hành động (Node)':<18} | {'F (Frontier - Priority Queue)':<35} | {'R (Reached)':<30} | Ghi chú")
    print("-" * 115)
    print_row_ucs("(Khởi tạo)", frontier, reached, "Khởi tạo nút gốc S với g=0")
    while frontier:
        g_cost, current_node, path = heapq.heappop(frontier)
        node_name = state_to_str(current_node)
        print_row_ucs(f"POP {node_name}({g_cost})", frontier, reached, f"Lấy nút có g nhỏ nhất")
        if current_node == GOAL_STATE:
            print_row_ucs(f"Check {node_name}", frontier, reached, "-> TRÙNG GOAL! THÀNH CÔNG.")
            return path, g_cost
        print_row_ucs(f"Check {node_name}", frontier, reached, f"-> {node_name} != GOAL")
        has_child = False
        for child, edge_cost in get_ucs_successors_by_misplaced(current_node):
            new_g = g_cost + edge_cost
            if child not in reached or new_g < reached[child]:
                reached[child] = new_g
                heapq.heappush(frontier, (new_g, child, path + [child]))
                has_child = True
        if has_child:
            print_row_ucs("(Nạp con vào F)", frontier, reached, f"Nạp con kèm chi phí (edge_cost = số ô sai)")
        print("-" * 115)
    return None, float('inf')
print("=== BẢNG TRACE UCS (CHI PHÍ = SỐ Ô SAI) ===")
ucs_path, total_cost = ucs_misplaced_trace(START_STATE)
print("\n" + "="*50)
print(f"TỔNG CHI PHÍ Ô SAI TỐI ƯU: {total_cost}")
print("="*50 + "\n")
print("CHI TIẾT CÁC BƯỚC DỊCH CHUYỂN TỐI ƯU:")
for idx, st in enumerate(ucs_path):
    print(f"Bước {idx} (Số ô sai hiện tại = {count_misplaced_tiles(st)}):")
    print_matrix(st)

=== BẢNG TRACE UCS (CHI PHÍ = SỐ Ô SAI) ===
-------------------------------------------------------------------------------------------------------------------
Hành động (Node)   | F (Frontier - Priority Queue)       | R (Reached)                    | Ghi chú
-------------------------------------------------------------------------------------------------------------------
(Khởi tạo)         | [S(0)]                              | {S:0}                          | Khởi tạo nút gốc S với g=0
POP S(0)           | []                                  | {S:0}                          | Lấy nút có g nhỏ nhất
Check S            | []                                  | {S:0}                          | -> S != GOAL
(Nạp con vào F)    | [N_7(1), N_1(3), N_3(3), N_5(3)]    | {S:0, N_1:3, N_7:1, N_3:3, N_5:3} | Nạp con kèm chi phí (edge_cost = số ô sai)
-------------------------------------------------------------------------------------------------------------------
POP N_7(1)         | [N_1(3), N_